# Module B03 — Making Decisions

## Exercise 5: Every visitor gets a price

A ticket desk needs a price for whoever walks up. Not most people: everyone. A
program that handles the common cases and falls silent on the rest is the one
that gets a phone call on a Sunday.

This notebook is about coverage. You will write a price that depends on two
things at once, check that every possible input lands somewhere, and meet
`match`, a fourth way of choosing a branch.

Prices are in whole pence throughout, because module B02 showed what floating
point does to money.

| | |
|---|---|
| Time | About 45 minutes |
| You need | This notebook |
| Comes after | Exercise 4, four levels deep, then flat |

---

## 1. What you are carrying in

An `elif` chain asks one question with several answers. The first true branch
runs and the rest are skipped, so the order decides the result. `else` catches
everything the chain missed. `and`, `or`, and `not` combine conditions, and
brackets say where the grouping goes.

One run, so this notebook stands on its own.

In [ ]:
age = 30

if age >= 65:
    band = "senior"
elif age >= 18:
    band = "adult"
else:
    band = "child"

print(band)

---

## 2. Chained comparisons

A band has two edges. Written the long way, that is two comparisons joined by
`and`.

```python
age >= 5 and age < 18
```

Python lets you write the same thing the way it appears in mathematics, with the
value in the middle.

In [ ]:
age = 12

print("the long way: ", age >= 5 and age < 18)
print("chained:      ", 5 <= age < 18)

```
5 <= age < 18
│    │  │   │
│    │  │   └── upper edge, excluded, because < not <=
│    │  └── the value being tested, written once
│    └── the first comparison
└── lower edge, included, because <= not <
```

Read it as one sentence: age is at least 5 and below 18.

Three details.

- Any comparison operators work, in any mix: `0 < x <= 100`, `a == b == c`.
- The middle value is evaluated once, which matters when it is expensive to
  produce.
- The chain is still an `and`, so short-circuiting applies exactly as exercise 3
  described.

Chained comparisons are shorter, and they put the two edges where a reader can
compare them at a glance. That is what makes an off-by-one visible.

---

## 3. The edges are the whole job

Write the table before the code. It takes two minutes and it is where the
mistakes are.

| Age | Band | Price |
|---|---|---|
| under 5 | infant | 0 |
| 5 to 17 | child | 600 |
| 18 to 64 | adult | 1200 |
| 65 and over | senior | 800 |

Every edge in that table is a decision about `<` against `<=`. A 5 year old is a
child, not an infant. A 65 year old is a senior, not an adult.

In [ ]:
age = 5

if age < 5:
    price = 0
elif 5 <= age < 18:
    price = 600
elif 18 <= age < 65:
    price = 1200
else:
    price = 800

print("age", age, "pays", price, "pence")

Test the edges, not the middle. An age of 30 gets the right price from almost
any version of this code, including several wrong ones. The ages worth running
are 4, 5, 17, 18, 64, and 65.

Note also that once the chain is ordered, the lower edge of each band is
guaranteed by the branch above it. `elif age < 18:` would do here, because
anything below 5 was already taken. Writing `5 <= age < 18` states the band in
full and survives somebody reordering the chain later. Both are defensible;
being deliberate is not optional.

---

## 4. What happens to an input nobody thought about

The chain above has an `else`, so every age lands somewhere. Take the `else`
away and see what the program does with a value that matches nothing.

The next cell fails on purpose. It uses the name `ticket_price`, which nothing
in this notebook has set. That detail is the whole point, and the prose after it
explains why.

In [ ]:
age = 70

if age < 5:
    ticket_price = 0
elif 5 <= age < 18:
    ticket_price = 600
elif 18 <= age < 65:
    ticket_price = 1200

print("age", age, "pays", ticket_price, "pence")

```
NameError: name 'ticket_price' is not defined
```

**`NameError`** is the error type: a name was used that nothing has been
attached to. The cause is that no branch matched, so no branch ran, so
`ticket_price` was never assigned.

This traceback is the lucky version, and it is lucky only because the name was
fresh. Change `ticket_price` back to `price` in that cell and run it again: the
`price` left over from section 3 is still attached, so a 70 year old is charged
600 pence, no error appears, and the number is wrong. That is exercise 1's
silent failure, arriving through a different door.

**A chain that assigns something needs a final `else`.** If there is genuinely
no sensible default, the `else` should say so out loud rather than be left off.

In [ ]:
age = -3

if age < 0:
    price = None
    note = "an age below zero is not a valid ticket"
elif age < 5:
    price = 0
    note = "infant"
elif 5 <= age < 18:
    price = 600
    note = "child"
elif 18 <= age < 65:
    price = 1200
    note = "adult"
else:
    price = 800
    note = "senior"

print("age", age, ":", note, "price", price)

`None`, from exercise 2, is the right value for "there is no price", and it is
falsy, so `if price:` would treat it as absent. That is a case where `is None`
is the check you want.

---

## 5. Two dimensions: age and day

Tuesday is half price for everyone. That is a second rule about a different
thing, so it belongs in a second decision rather than inside the first one.

In [ ]:
age = 30
day = "tuesday"

if age < 5:
    price = 0
elif 5 <= age < 18:
    price = 600
elif 18 <= age < 65:
    price = 1200
else:
    price = 800

if day == "tuesday":
    price = price // 2

print(price, "pence on", day)

Two chains, one after the other, each about one thing. The alternative is a
branch for every combination of age band and day, which is four times seven
branches, and every one of them a place to make a typing mistake.

`//` is floor division from module B02. Working in whole pence means the half
price of 600 is exactly 300, with no fraction to lose.

---

## 6. `match`, a first look

When the decision is "this one value against a short list of fixed
possibilities", an `elif` chain repeats the value on every line.

```python
if day == "saturday":
    ...
elif day == "sunday":
    ...
elif day == "tuesday":
    ...
```

`match` says the value once.

In [ ]:
day = "saturday"

match day:
    case "saturday":
        surcharge = 200
    case "sunday":
        surcharge = 200
    case _:
        surcharge = 0

print("surcharge:", surcharge, "pence")

```
match day:
│     └── the value being matched, written once
└── the keyword

    case "saturday":
    │    └── a pattern. Here, a literal to compare against
    └── the keyword

        surcharge = 200
        └── the block, indented as usual

    case _:
         └── the underscore matches anything. It is the else of a match
```

Two cases producing the same result can share a line with `|`, which reads as
"or".

In [ ]:
day = "sunday"

match day:
    case "saturday" | "sunday":
        surcharge = 200
    case _:
        surcharge = 0

print("surcharge:", surcharge, "pence")

Three things to hold on to, and one to avoid.

- `match` runs the first case that matches and skips the rest, exactly like an
  `elif` chain.
- `case _:` is the catch-all. Without it, a value matching nothing runs nothing,
  and you are back in section 4.
- `match` needs Python 3.10 or newer. On an older version it is a `SyntaxError`.
- `match` is far more capable than this. It can take patterns apart, bind pieces
  of them to names, and test conditions on the parts. **Intermediate module 04
  makes it precise.** Today, use it for one value against a few fixed literals,
  which is what it is best at and where it cannot surprise you.

That last point has a trap attached, and it is worth meeting now.

---

## 7. The `match` trap: a bare name does not compare

A pattern that is a plain name does not compare against that name's value. It
**captures**: it matches anything at all and attaches the name to it.

Somebody storing the weekend days in names and matching against them expects a
comparison. The next cell fails on purpose. As before, the code is held in text
and handed to `exec`, because a cell containing a syntax error cannot run at
all.

In [ ]:
trap = '''SATURDAY = "saturday"

day = "wednesday"

match day:
    case SATURDAY:
        print("weekend")
    case "sunday":
        print("also weekend")
    case _:
        print("weekday")
'''

exec(trap)

```
SyntaxError: name capture 'SATURDAY' makes remaining patterns unreachable
```

**`SyntaxError`** is the error type. The cause is that `case SATURDAY:` is a
capture pattern, not a comparison. It matches every value, so the two cases
below it can never run, and Python refuses the whole thing rather than let you
ship it.

Python catching this is a kindness. Written with the capture as the only case,
there is nothing unreachable and no error, and the code silently reports every
day of the week as the weekend.

For today, keep the literals in the `case` lines, as section 6 does. Intermediate
module 04 covers the form that compares against a name, and the rule that decides
which behaviour you get.

---

# Your turn

**Do not delete the `# ANSWER n` marker lines.**

### Task 1

Using the table from section 3 and the Tuesday half-price rule from section 5,
predict the price in pence for each visitor **before running anything**.

In [ ]:
# ANSWER 1
# age 4,  monday   -> ___
# age 5,  monday   -> ___
# age 17, tuesday  -> ___
# age 18, monday   -> ___
# age 64, tuesday  -> ___
# age 65, monday   -> ___

print("predictions written above. Task 2 builds the code that checks them.")

### Task 2

Write the age bands using chained comparisons. Fill in every blank, including
the final `else`.

Then run it for ages 4, 5, 17, 18, 64, and 65 and compare against your
predictions.

In [ ]:
# ANSWER 2
age = 17

if ___:
    price = 0
elif ___:
    price = 600
elif ___:
    price = 1200
else:
    price = ___

print("age", age, "pays", price, "pence")

### Task 3

Add the day rule with `match`.

Tuesday is half price. Saturday and Sunday carry a surcharge of 200 pence. Every
other day is the plain price.

Keep the literals in the `case` lines, for the reason section 7 gave.

In [ ]:
# ANSWER 3
age = 30
day = "saturday"

if age < 5:
    price = 0
elif 5 <= age < 18:
    price = 600
elif 18 <= age < 65:
    price = 1200
else:
    price = 800

match day:
    case "tuesday":
        price = ___
    case "___" | "___":
        price = price + 200
    case _:
        price = price

print(age, "on", day, "pays", price, "pence")

### Task 4

Coverage. The code below has a gap, and the gap is not where the missing `else`
was in section 4.

Run it as it stands for each of these ages: 0, 4, 5, 17, 18, 64, 65, and 100.
Exactly one of them lands in the wrong band and is charged the wrong price. No
traceback appears.

Find it, fix it, and write down which age exposed it and what the wrong price
was.

In [ ]:
# ANSWER 4
age = 0

if age < 5:
    price = 0
elif 5 < age < 18:
    price = 600
elif 18 <= age < 65:
    price = 1200
else:
    price = 800

print("age", age, "pays", price, "pence")

# Which age exposed the gap? ___
# What price did it get, and what should it have got? ___
# Why did no traceback appear? ___

### Task 5

Put it together into something you would show somebody: a till receipt.

Print the age, the day, the band the visitor fell into, and the price in pence.
Cover every age from below zero upwards, and every day.

Then answer the judgement question in the comment: for the day rule, would you
maintain the `match` or an `elif` chain, and what would change your mind.

In [ ]:
# ANSWER 5
age = 70
day = "tuesday"

# 1. band and base price, covering every age including negative ones
if ___:
    band = "not a valid age"
    price = None
elif ___:
    band = "infant"
    price = 0
elif ___:
    band = "child"
    price = 600
elif ___:
    band = "adult"
    price = 1200
else:
    band = "senior"
    price = 800

# 2. the day rule, only when there is a price to adjust
if price is not None:
    match day:
        case "tuesday":
            price = ___
        case "saturday" | "sunday":
            price = ___
        case _:
            price = price

# 3. the receipt
print("age: ", age)
print("day: ", day)
print("band:", band)
print("price:", price, "pence")

match_or_elif_for_the_day_rule = "___"
what_would_change_my_mind = "___"

---

## Self-check

You do not need to understand this cell. It is machinery, not material.

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "That is module B03 complete."

a1, a2, a3, a4, a5 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                      _answer("# ANSWER 3"), _answer("# ANSWER 4"),
                      _answer("# ANSWER 5"))

results = [
    check("monday   -> ___" not in a1 and a1.count("___") == 0,
          "Task 1: all six prices predicted before running"),
    check("___" not in a2, "Task 2: every band and the final else filled in"),
    check(a2.count("<") >= 4, "Task 2: the bands use chained comparisons"),
    check("match" in a3 and "case _" in a3 and "___" not in a3,
          "Task 3: the day rule uses match with a catch-all case"),
    check("5<=age" in a4.replace(" ", "") or "age>=5" in a4.replace(" ", ""),
          "Task 4: the child band now includes its lower edge"),
    check("exposed the gap? ___" not in a4 and "no traceback appear? ___" not in a4,
          "Task 4: you named the age, the wrong price, and why it was silent"),
    check("___" not in a5 and "band" in a5 and "price" in a5,
          "Task 5: the receipt prints a band and a price for every case"),
    check("change_my_mind" in a5 and a5.count('"___"') == 0,
          "Task 5: you chose match or elif and said what would change your mind"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- A chained comparison such as `5 <= age < 18` states a band in one expression,
  puts both edges where they can be compared, and evaluates the middle value
  once.
- The edges are where the mistakes are. Test 4, 5, 17, 18, 64, and 65, not 30.
- A chain that assigns something needs a final `else`. Without one, an
  unmatched input either raises a `NameError` or, worse, leaves a stale value in
  place and says nothing.
- `None` is the honest value for "there is no price", and `is None` is how you
  test for it.
- Two independent rules belong in two decisions, one after the other, not in one
  branch per combination.
- `match` states the value once and compares it against literal patterns, with
  `|` for alternatives and `case _:` as the catch-all.
- A bare name in a `case` captures rather than compares. Intermediate module 04
  makes `match` precise; until then, keep literals in the `case` lines.

## Before you move on

- [ ] You ran a chain with no `else` and read the `NameError` it produced.
- [ ] You found the age that the broken band in task 4 mispriced.
- [ ] You wrote a `match` with a catch-all case.
- [ ] You can say why a bare name in a `case` line is a trap.
- [ ] Your receipt gives a sensible answer for an age of -3.

**Next:** module B04, where the decisions you have written here start repeating,
once for every item you are given.